In [1]:
# Import packages and modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

In [2]:
# Check the version of XGBoost
print("Found XGBoost v" + xgboost.__version__)

Found XGBoost v2.1.4


In [3]:
datasetPath = "dataset/task2/"

In [4]:
# Parameters
datasetPath = "dataset/577p/gldm/kernel5/"


In [5]:
# train/validation/test = 70/10/20
train_data = pd.read_csv(f"{datasetPath}dataset.train.csv")
# validation_data = pd.read_csv(f"{datasetPath}dataset.validation.csv")
# test_data = pd.read_csv(f"{datasetPath}dataset.test.csv")

## 5-fold cross-validation at patient level

Cross-validation is performed only on the development set (**Train + Validation**). The independent **Test** set is not used during cross-validation or model selection. All voxels from the same patient are kept in the same fold by `StratifiedGroupKFold`.


In [6]:
# 5-fold patient-level cross-validation (Train + Validation only)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix
)

# Detect the patient identifier column. Add another candidate here if your CSV uses a different name.
group_candidates = [
    "patient_id", "patient", "case_id", "case",
    "subject_id", "subject", "BraTS21ID", "brats_id"
]
group_col = next((c for c in group_candidates if c in train_data.columns), None)
if group_col is None:
    raise ValueError(
        "Patient ID column was not found. "
        "Set group_col manually to the column identifying each patient, "
        "for example: group_col = 'patient_id'."
    )

print(f"Patient/group column: {group_col}")

# IMPORTANT: keep the independent test set completely outside CV.
dev_data = pd.concat([train_data], ignore_index=True)

# Exclude labels, weights, and patient/case metadata from model features.
metadata_cols = set(group_candidates + ["label", "sample_weight"])
cv_feature_cols = [c for c in dev_data.columns if c not in metadata_cols]

X_dev = dev_data[cv_feature_cols]
y_dev = dev_data["label"].astype(int)
groups_dev = dev_data[group_col]

# Sanity check: no patient in development data may occur in the independent test set.
# if group_col in test_data.columns:
#     overlap = set(groups_dev.unique()) & set(test_data[group_col].unique())
#     assert len(overlap) == 0, (
#         f"Data leakage: {len(overlap)} patient(s) occur in both development and test sets: "
#         f"{list(overlap)[:10]}"
#     )

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_dev, y_dev, groups=groups_dev), start=1
):
    X_fold_train = X_dev.iloc[train_idx]
    y_fold_train = y_dev.iloc[train_idx]
    X_fold_val = X_dev.iloc[val_idx]
    y_fold_val = y_dev.iloc[val_idx]

    train_groups = set(groups_dev.iloc[train_idx])
    val_groups = set(groups_dev.iloc[val_idx])
    assert train_groups.isdisjoint(val_groups), f"Patient leakage in fold {fold}"

    # Compute class weights from this fold's training partition only.
    fold_classes = np.unique(y_fold_train)
    fold_weights = compute_class_weight(
        class_weight="balanced",
        classes=fold_classes,
        y=y_fold_train
    )
    fold_class_weight = dict(zip(fold_classes, fold_weights))
    fold_sample_weight = y_fold_train.map(fold_class_weight).to_numpy()

    cv_model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric=["logloss", "auc"],
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    cv_model.fit(
        X_fold_train,
        y_fold_train,
        sample_weight=fold_sample_weight,
        verbose=False
    )

    fold_prob = cv_model.predict_proba(X_fold_val)[:, 1]
    fold_pred = (fold_prob >= 0.5).astype(int)

    cm = confusion_matrix(y_fold_val, fold_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    result = {
        "fold": fold,
        "train_patients": len(train_groups),
        "val_patients": len(val_groups),
        "train_voxels": len(train_idx),
        "val_voxels": len(val_idx),
        "accuracy": accuracy_score(y_fold_val, fold_pred),
        "precision": precision_score(y_fold_val, fold_pred, zero_division=0),
        "recall": recall_score(y_fold_val, fold_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_fold_val, fold_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_fold_val, fold_prob),
        "pr_auc": average_precision_score(y_fold_val, fold_prob),
        "mcc": matthews_corrcoef(y_fold_val, fold_pred),
    }
    fold_results.append(result)

    print(
        f"Fold {fold}: "
        f"patients(train/val)={len(train_groups)}/{len(val_groups)}, "
        f"AUC={result['roc_auc']:.4f}, F1={result['f1']:.4f}, "
        f"Recall={result['recall']:.4f}"
    )

cv_results = pd.DataFrame(fold_results)
display(cv_results)

metric_cols = [
    "accuracy", "precision", "recall", "specificity",
    "f1", "roc_auc", "pr_auc", "mcc"
]
cv_summary = pd.DataFrame({
    "mean": cv_results[metric_cols].mean(),
    "std": cv_results[metric_cols].std(ddof=1),
})

print("\n5-fold patient-level CV summary (mean ± SD):")
for metric in metric_cols:
    print(
        f"{metric:12s}: "
        f"{cv_summary.loc[metric, 'mean']:.4f} ± "
        f"{cv_summary.loc[metric, 'std']:.4f}"
    )

display(cv_summary)

Patient/group column: patient_id


Fold 1: patients(train/val)=461/116, AUC=0.9532, F1=0.5062, Recall=0.8495


Fold 2: patients(train/val)=462/115, AUC=0.9322, F1=0.4202, Recall=0.8598


Fold 3: patients(train/val)=463/114, AUC=0.9495, F1=0.5072, Recall=0.8657


Fold 4: patients(train/val)=460/117, AUC=0.9326, F1=0.4462, Recall=0.8253


Fold 5: patients(train/val)=462/115, AUC=0.9429, F1=0.4767, Recall=0.8594


,fold,train_patients,val_patients,train_voxels,val_voxels,accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,mcc
0,1,461,116,521577,131820,0.899545,0.360546,0.849456,0.902777,0.506227,0.953158,0.700338,0.513036
1,2,462,115,523132,130265,0.851265,0.278020,0.859767,0.850696,0.420171,0.932165,0.644048,0.435634
2,3,463,114,523605,129792,0.885686,0.358653,0.865744,0.887140,0.507191,0.949461,0.713730,0.511672
3,4,460,117,521845,131552,0.869428,0.305731,0.825262,0.872434,0.446171,0.932643,0.630979,0.451579
4,5,462,115,523429,129968,0.880463,0.329853,0.859381,0.881889,0.476726,0.942867,0.687907,0.486415



5-fold patient-level CV summary (mean ± SD):
accuracy    : 0.8773 ± 0.0181
precision   : 0.3266 ± 0.0353
recall      : 0.8519 ± 0.0160
specificity : 0.8790 ± 0.0193
f1          : 0.4713 ± 0.0380
roc_auc     : 0.9421 ± 0.0096
pr_auc      : 0.6754 ± 0.0361
mcc         : 0.4797 ± 0.0350


,mean,std
accuracy,0.877277,0.018131
precision,0.326561,0.035305
recall,0.851922,0.016008
specificity,0.878987,0.019263
f1,0.471297,0.038024
roc_auc,0.942059,0.009555
pr_auc,0.675400,0.036068
mcc,0.479667,0.035040
